# Fine-tune `facebook/mms-tts-pcm` on the Honest Orb voice

Base recipe: [ylacombe/finetune-hf-vits](https://github.com/ylacombe/finetune-hf-vits) (`run_vits_finetuning.py` + JSON config, launched with `accelerate`).
Pidgin uses Latin script, so no `uroman` step is needed. The training voice is the **disguised** dataset (`data/voice/dataset_disguised/`, pitch-shifted −2 semitones).

**Budget:** ~20–45 min training on a free Colab T4 (128 train clips / batch 16 = 8 steps/epoch × 80 epochs = 640 steps at ~2–4 s/step, fp16), plus ~10 min setup. Total well under 1.5 h. Checkpoints are saved every 8 epochs so an interrupted run can resume.

Before you run:

1. Colab menu → Runtime → **Change runtime type → T4 GPU**.
2. Upload `voice_dataset.zip` (repo root; made from `data/voice/dataset_disguised/`: `metadata.csv` + `wavs/` at the top level) when Cell 1 asks.
3. Create a Hugging Face token with **write** access and save it as a Colab secret named `HF_TOKEN` (or log in when asked).


In [ ]:
# Cell 1. Upload the disguised dataset (voice_dataset.zip from the repo root).
from google.colab import files

up = files.upload()  # select voice_dataset.zip
assert any(n.endswith(".zip") for n in up), "upload voice_dataset.zip first"
!rm -rf /content/voice_dataset && mkdir -p /content/voice_dataset
!unzip -q /content/voice_dataset.zip -d /content/voice_dataset
!head -3 /content/voice_dataset/metadata.csv && echo "wavs:" && ls /content/voice_dataset/wavs | wc -l


In [ ]:
# Confirm we have a GPU (want to see a Tesla T4).
!nvidia-smi --query-gpu=name --format=csv,noheader


In [ ]:
# Recipe + pinned deps + the Cython monotonic-alignment op.
# Pins follow finetune-hf-vits' own requirements (transformers/datasets/acclerate
# floors the script was written against); transformers is capped below v5, whose
# trainer API the script does not support. The Cython op is required: the
# pure-Python alignment is far too slow.
!git clone https://github.com/ylacombe/finetune-hf-vits.git
%cd /content/finetune-hf-vits
!pip -q install "transformers>=4.35.1,<5" "datasets[audio]>=2.14.7,<3" "accelerate>=0.24.1" soundfile librosa tensorboard Cython
!cd monotonic_align && mkdir -p monotonic_align && python setup.py build_ext --inplace && cd ..


In [ ]:
# Hugging Face login (pushes the dataset + the finetuned model to your account).
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    from huggingface_hub import login
    login(token=token)
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()  # paste a token with write access

HANDLE = input('Your HF username (e.g. octopus): ').strip()  # used for repo names below


## Speech normalization (same as serving)

`api/tts.py` speaks every sentence through `normalize_for_speech` (numbers → words, `q` → `k`, symbols stripped), because the mms-tts-pcm tokenizer silently drops anything outside its 37-token vocab. The transcripts below are normalized with that **same function, copied verbatim**, so training sees exactly what the server will speak.


In [ ]:
import re

_ONES = [
    "zero", "one", "two", "three", "four", "five", "six", "seven",
    "eight", "nine", "ten", "eleven", "twelve", "thirteen", "fourteen",
    "fifteen", "sixteen", "seventeen", "eighteen", "nineteen",
]
_TENS = {
    2: "twenty", 3: "thirty", 4: "forty", 5: "fifty",
    6: "sixty", 7: "seventy", 8: "eighty", 9: "ninety",
}


def _num_words(n: int) -> str:
    """Cardinal words for a non-negative int (supports large values)."""
    if n < 20:
        return _ONES[n]
    if n < 100:
        tens, ones = divmod(n, 10)
        return _TENS[tens] if ones == 0 else f"{_TENS[tens]}-{_ONES[ones]}"
    if n < 1000:
        hundreds, rest = divmod(n, 100)
        tail = "" if rest == 0 else " " + _num_words(rest)
        return f"{_ONES[hundreds]} hundred{tail}"
    if n < 1_000_000:
        thousands, rest = divmod(n, 1000)
        tail = "" if rest == 0 else " " + _num_words(rest)
        return f"{_num_words(thousands)} thousand{tail}"
    return " ".join(_ONES[int(d)] for d in str(n))


_SECTION_RANGE_SS = re.compile(r"\bss\.?\s*(\d+)\s*[-\u2013\u2014]\s*(\d+)", re.IGNORECASE)
_SECTION_RANGE_S = re.compile(r"\bs\.\s*(\d+)\s*[-\u2013\u2014]\s*(\d+)", re.IGNORECASE)
_SECTION_SINGLE_SS = re.compile(r"\bss\.?\s*(\d+)", re.IGNORECASE)
_SECTION_SINGLE_S = re.compile(r"\bs\.\s*(\d+)", re.IGNORECASE)
_SECTION_BARE_S = re.compile(r"\bs(\d{1,4})\b", re.IGNORECASE)
_GENERIC_RANGE = re.compile(r"\b(\d+)\s*[-\u2013\u2014]\s*(\d+)")
_DECIMAL = re.compile(r"\b(\d+)\.(\d+)\b")
_ORDINAL = re.compile(r"\b(\d+)(?:st|nd|rd|th)\b", re.IGNORECASE)
_NUMBER_AFTER = re.compile(r"\d+")


def normalize_for_speech(text: str) -> str:
    """Rewrite text into words the mms-tts-pcm tokenizer can speak.

    Numbers become words (``35`` -> ``thirty-five``), the emergency
    number is read digit by digit (``112`` -> ``one one two``), legal
    short-hands expand (``s.35`` -> ``section thirty-five``,
    ``ss.49-51`` -> ``sections forty-nine to fifty-one``), brackets and
    symbols are stripped, and common abbreviations are expanded. The
    result only contains tokenizer-safe characters (``a``-``z`` except
    ``q``, space, apostrophe, hyphen); the missing letter ``q`` is mapped
    to its closest supported sound ``k``.
    """
    text = text.lower()

    # Legal section shorthands first (they consume their digits).
    text = _SECTION_RANGE_SS.sub(
        lambda m: f"sections {_num_words(int(m.group(1)))} to "
        f"{_num_words(int(m.group(2)))}",
        text,
    )
    text = _SECTION_RANGE_S.sub(
        lambda m: f"sections {_num_words(int(m.group(1)))} to "
        f"{_num_words(int(m.group(2)))}",
        text,
    )
    text = _SECTION_SINGLE_SS.sub(
        lambda m: f"sections {_num_words(int(m.group(1)))}", text
    )
    text = _SECTION_SINGLE_S.sub(
        lambda m: f"section {_num_words(int(m.group(1)))}", text
    )
    text = _SECTION_BARE_S.sub(
        lambda m: f"section {_num_words(int(m.group(1)))}", text
    )

    # Emergency number: read digit by digit, never "one hundred twelve".
    text = re.sub(r"\b112\b", "one one two", text)

    # Units and abbreviations.
    text = re.sub(r"(\d)\s*km\b", r"\1 kilometers", text)
    text = text.replace("km", "kilometers")
    text = re.sub(r"\be\.g\.", "for example", text)
    text = re.sub(r"\bi\.e\.", "that is", text)
    text = re.sub(r"\betc\.?", "etcetera", text)
    text = re.sub(r"\bmrs\.?(?!\w)", "missus", text)
    text = re.sub(r"\bmr\.?(?!\w)", "mister", text)
    text = re.sub(r"\bdr\.?(?!\w)", "doctor", text)
    text = re.sub(r"\bvs\.?(?!\w)", "versus", text)
    text = re.sub(
        r"\bno\.\s*(\d+)", lambda m: f"number {_num_words(int(m.group(1)))}", text
    )
    text = text.replace("§", " section ")
    text = text.replace("&", " and ")
    text = text.replace("%", " percent ")
    text = text.replace("+", " plus ")
    text = text.replace("=", " equals ")
    text = text.replace("@", " at ")
    text = text.replace("/", " or ")
    text = text.replace("₦", " naira ")
    text = re.sub(r"\bngn\b", "naira", text)
    text = text.replace("$", " dollars ")
    text = text.replace("°", " degrees ")

    # Numbers: strip thousand separators, read decimals digit by digit
    # after the point, ordinals as cardinals, then plain integers.
    text = re.sub(r"(?<=\d),(?=\d)", "", text)
    text = _DECIMAL.sub(
        lambda m: f"{_num_words(int(m.group(1)))} point "
        + " ".join(_ONES[int(d)] for d in m.group(2)),
        text,
    )
    text = _ORDINAL.sub(lambda m: _num_words(int(m.group(1))), text)
    text = _GENERIC_RANGE.sub(
        lambda m: f"{_num_words(int(m.group(1)))} to "
        f"{_num_words(int(m.group(2)))}",
        text,
    )
    text = _NUMBER_AFTER.sub(lambda m: _num_words(int(m.group())), text)

    # Missing vocab letter -> closest supported sound.
    text = text.replace("q", "k")

    # Drop anything the tokenizer would silently swallow anyway.
    text = re.sub(r"[^a-z' \-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Quick check the copy works:
print(normalize_for_speech("Call 112! S.35 and s.49-51 protect you. Q is rare."))



In [ ]:
# Wrap wavs + normalized transcripts as a Hub dataset (train/test split;
# test doubles as the eval split). 137 clips -> 128 train / 9 eval.
import pandas as pd
from datasets import Dataset, Audio

df = pd.read_csv('/content/voice_dataset/metadata.csv', sep='|')  # columns: file, text
print('rows:', len(df))
df['text'] = df['text'].apply(normalize_for_speech)
print('example:', df['text'].iloc[0])
df['audio'] = df['file'].apply(lambda f: f'/content/voice_dataset/{f}')
ds = Dataset.from_pandas(df[['audio', 'text']]).cast_column('audio', Audio(sampling_rate=16000))
splits = ds.train_test_split(test_size=9, seed=42)
print(splits)
splits.push_to_hub(f'{HANDLE}/honest-orb-pcm-voice', private=True)


## Build the Pidgin training checkpoint

There is no ready-made `mms-tts-pcm-train` checkpoint (only eng/spa/kor/mar/tam/guj exist), so per the recipe's Option 2 we convert the discriminator of `facebook/mms-tts-pcm` once. Keep this checkpoint: re-running finetunes later skips this step.


In [ ]:
# '--push_to_hub' also saves it remotely; drop that flag to keep it local-only
!python convert_original_discriminator_checkpoint.py --language_code pcm \
  --pytorch_dump_folder_path /content/mms-tts-pcm-train \
  --push_to_hub {HANDLE}/mms-tts-pcm-train


## Finetune (the only long cell: ~20–45 min on a T4)

Config keeps the recipe's loss weights / optimizer, sized for 128 train clips: batch 16 → 8 steps/epoch, 80 epochs → 640 steps with fp16. Eval + checkpoint every 8 epochs; the last 3 checkpoints are kept, and a re-run auto-resumes from the newest one, so watch eval loss and stop early if it climbs (small-data overfitting).


In [ ]:
import json

config = {
    'project_name': 'honest_orb_pcm',
    'push_to_hub': True,
    'hub_model_id': f'{HANDLE}/honest-orb-mms-pcm',
    'overwrite_output_dir': True,
    'output_dir': '/content/honest-orb-pcm',
    'dataset_name': f'{HANDLE}/honest-orb-pcm-voice',
    'audio_column_name': 'audio',
    'text_column_name': 'text',
    'train_split_name': 'train',
    'eval_split_name': 'test',
    'max_duration_in_seconds': 20,
    'min_duration_in_seconds': 0.5,
    'max_tokens_length': 500,
    'model_name_or_path': f'{HANDLE}/mms-tts-pcm-train',
    'preprocessing_num_workers': 2,
    'do_train': True,
    'num_train_epochs': 80,
    'gradient_accumulation_steps': 1,
    'gradient_checkpointing': False,
    'per_device_train_batch_size': 16,
    'learning_rate': 2e-5,
    'adam_beta1': 0.8,
    'adam_beta2': 0.99,
    'warmup_ratio': 0.01,
    'group_by_length': False,
    'do_eval': True,
    'eval_steps': 64,
    'per_device_eval_batch_size': 8,
    'max_eval_samples': 9,
    'do_step_schedule_per_epoch': True,
    'weight_disc': 3,
    'weight_fmaps': 1,
    'weight_gen': 1,
    'weight_kl': 1.5,
    'weight_duration': 1,
    'weight_mel': 35,
    'fp16': True,
    'save_steps': 64,
    'save_total_limit': 3,
    'logging_steps': 16,
    'report_to': 'tensorboard',
    'seed': 42,
}
with open('/content/honest_orb_pcm.json', 'w') as f:
    json.dump(config, f, indent=4)
print('wrote /content/honest_orb_pcm.json (~640 steps total)')


In [ ]:
!accelerate launch run_vits_finetuning.py /content/honest_orb_pcm.json


## Test it and take it home

`push_to_hub: true` already uploaded the model to `{HANDLE}/honest-orb-mms-pcm`. Listen to the 3 lines below first — if they sound right, the zip in the next cell is the local copy: unzip it next to the repo and point `api/tts.py`'s `MODEL_ID` at that folder (it loads via `AutoTokenizer`/`VitsModel`, same as today).


In [ ]:
# Listen: 2 Pidgin + 1 English, normalized exactly like the server does.
from transformers import pipeline
import scipy.io.wavfile
from IPython.display import Audio, display

synth = pipeline('text-to-speech', f'{HANDLE}/honest-orb-mms-pcm')
tests = [
    ('pidgin-1', 'Abeg officer, show me the warrant before you search my car.'),
    ('pidgin-2', 'No wahala, Honest Orb go guide you step by step today.'),
    ('english-1', 'You have the right to remain silent until your lawyer arrives.'),
]
for name, sentence in tests:
    out = synth(normalize_for_speech(sentence))
    path = f'/content/honest_orb_test_{name}.wav'
    scipy.io.wavfile.write(path, rate=out['sampling_rate'], data=out['audio'])
    print(sentence)
    display(Audio(path))


In [ ]:
# Take it home: zip the finetuned model and download it.
!rm -f /content/honest_orb_voice.zip && cd /content && zip -qr honest_orb_voice.zip honest-orb-pcm
from google.colab import files
files.download('/content/honest_orb_voice.zip')


## Notes

- Nothing is trained until you run the `accelerate launch` cell; everything before it only prepares data and config.
- Checkpoints land in `/content/honest-orb-pcm/checkpoint-N` (last 3 kept). If training is interrupted, just re-run the launch cell: it auto-resumes from the newest checkpoint.
- If eval loss climbs while train loss keeps falling, stop early and take an earlier checkpoint — that is overfitting on 9 minutes of audio.
- License: `facebook/mms-tts-pcm` is CC BY-NC 4.0, and the finetuned voice inherits it — non-commercial use only, same as the base model.
